In [15]:
import sys
import os
import requests
import pandas as pd
from io import BytesIO
import streamlit as st
from datetime import datetime, time
import altair as alt

st.set_page_config(layout="wide")

sys.path.append(os.path.join(os.getcwd(), "..", "data"))

live_urls = [ "https://1drv.ms/x/c/d6aca2526f83594b/IQAMcIopdLU9SINiVRsgBHwWAZsXQJ5-PHSv1BevGvZ8f0Q?download=1",
              "https://1drv.ms/x/c/d6aca2526f83594b/IQAlE6FfCDS_QL0HXfdxWdtCAae3Ilx-a_K9hA_PXGN3Ofs?download=1"
               ]

urls = [ "https://1drv.ms/x/c/f6261b79731e452c/IQBpBPUwwVtQTYrz4LgAWE6ZAfTxxxhud5AzyDm4tVY5P-0?download=1",   # Dirty 1 - 24
         "https://1drv.ms/x/c/f6261b79731e452c/IQDTm4wgClDgRKutu40S_fAUASzQrkvm2Z--Qe2whPBX1xI?download=1"
         ]   # Dirty 25 - 107 

col = ['Date', 'Time', 'GPM_1', 'TOTAL_GAL_1', 'GPM_2','TOTAL_GAL_2', 'GPM_3', 'TOTAL_GAL_3', 'GPM_4', 'TOTAL_GAL_4', 'GPM_5','TOTAL_GAL_5', 'GPM_6', 'TOTAL_GAL_6', 'GPM_7', 'TOTAL_GAL_7', 'GPM_8','TOTAL_GAL_8', 'GPM_9', 'TOTAL_GAL_9', 'GPM_10', 'TOTAL_GAL_10', 'GPM_11','TOTAL_GAL_11', 'GPM_12', 'TOTAL_GAL_12', 'GPM_13', 'TOTAL_GAL_13','GPM_14', 'TOTAL_GAL_14', 'GPM_15', 'TOTAL_GAL_15', 'GPM_16','TOTAL_GAL_16', 'GPM_17', 'TOTAL_GAL_17', 'GPM_18', 'TOTAL_GAL_18','GPM_19', 'TOTAL_GAL_19', 'GPM_20', 'TOTAL_GAL_20', 'GPM_21','TOTAL_GAL_21', 'GPM_22', 'TOTAL_GAL_22', 'GPM_23', 'TOTAL_GAL_23','GPM_24', 'TOTAL_GAL_24', 'comments', 'datetime']
col2 = ['Date', 'Time','GPM_25', 'TOTAL_GAL_25', 'GPM_26','TOTAL_GAL_26','GPM_27', 'TOTAL_GAL_27', 'GPM_28','TOTAL_GAL_28', 'GPM_29', 'TOTAL_GAL_29', 'GPM_30','TOTAL_GAL_30', 'GPM_31', 'TOTAL_GAL_31', 'GPM_32', 'TOTAL_GAL_32', 'GPM_33','TOTAL_GAL_33', 'GPM_34', 'TOTAL_GAL_34', 'GPM_35', 'TOTAL_GAL_35', 'GPM_36','TOTAL_GAL_36', 'GPM_37', 'TOTAL_GAL_37', 'GPM_38', 'TOTAL_GAL_38', 'GPM_39','TOTAL_GAL_39', 'GPM_40', 'TOTAL_GAL_40', 'GPM_41', 'TOTAL_GAL_41','GPM_42', 'TOTAL_GAL_42', 'GPM_43', 'TOTAL_GAL_43', 'GPM_44','TOTAL_GAL_44', 'GPM_45', 'TOTAL_GAL_45', 'GPM_101', 'TOTAL_GAL_101','GPM_102', 'TOTAL_GAL_102', 'GPM_103', 'TOTAL_GAL_103', 'GPM_104','TOTAL_GAL_104', 'GPM_105', 'TOTAL_GAL_105', 'GPM_106', 'TOTAL_GAL_106','GPM_107', 'TOTAL_GAL_107', 'comments', 'datetime']

def get_data(url):
    response = requests.get(url)
    df = pd.read_excel(BytesIO(response.content), engine="openpyxl")
    return df

def shape_df(df):
    # cut the first 5 rows and first column 
    df = df.iloc[5:,1:].reset_index(drop=True)
    return df

def trim_frame(df, new_col):
    df = df.iloc[:, :len(new_col)] 
    df.columns = new_col
    return df

def clean_col(df):
    # strip all columns with "Unnamed" string
    for col in df.columns:
        if "Unnamed" in str(col):
            df = df.drop(columns=[str(col)], errors='ignore')
    return df

def fix_time(df):
    mask = df['Time'].apply(lambda x: isinstance(x, datetime))
    df.loc[mask, 'Time'] = df.loc[mask, 'Time'].apply(lambda x: x.time())
    return df

def make_datetime_col(df):
    df['datetime'] = df.apply(
        lambda row: datetime.combine(row['Date'].date(), row['Time']),
        axis=1
    )
    return df

def fix_space(df):
    df.columns = df.columns.str.replace('.', '_', regex=False)
    df.columns = df.columns.str.replace(' ', '_', regex=False)  # replace spaces
    return df

def drop_DT(df):
    # strip all columns with "Unnamed" string
    df = df.drop(columns=['Date', 'Time'], errors='ignore')
    return df

@st.cache_data()
def merge_and_sort(df1, df2):
    df_merged = pd.merge(df1, df2, on='datetime', how='outer', suffixes=('_1', '_2'))
    df_merged.sort_values('datetime', inplace=True)
    df_merged.reset_index(drop=True, inplace=True)
    df_merged = df_merged.drop(columns=['comments_1', 'comments_2'], errors='ignore')
    return df_merged

@st.cache_data()
def make_tidy(df):

    value_cols = [col for col in df.columns if 'GPM_' in col or 'TOTAL_GAL_' in col]

    df_long = df.melt(
        id_vars='datetime',
        value_vars=value_cols,
        var_name='variable',
        value_name='value'
    )

    # split column names
    df_long[['metric', 'pump']] = df_long['variable'].str.extract(r'(GPM|TOTAL_GAL)_(\d+)')
    df_long['pump'] = df_long['pump'].astype(int)

    # 🔑 FIX: force numeric
    df_long['value'] = pd.to_numeric(df_long['value'], errors='coerce')

    # 🔑 apply rule: GPM < 1 → 0
    df_long.loc[(df_long['metric'] == 'GPM') & (df_long['value'] < 1),'value'] = 0
    # pivot
    df_tidy = df_long.pivot_table(
        index=['datetime', 'pump'],
        columns='metric',
        values='value'
    ).reset_index()
    df_tidy.columns.name = None
    return df_tidy

@st.cache_data()
def run_functions(url, list):
    df = get_data(url)
    df = shape_df(df)
    df = trim_frame(df, list)  # df2 check too 
    df = clean_col(df)
    df = fix_time(df)
    df = make_datetime_col(df)
    df = fix_space(df)
    df = drop_DT(df)
    print("The functions ran ....")
    return df

@st.cache_data()
def get_daily_volume(df_tidy):
    df_tidy['date'] = df_tidy['datetime'].dt.date
    daily_total = (
        df_tidy.groupby(['date', 'pump'])['TOTAL_GAL']
        .agg(day_start='first', day_end='last')
        .reset_index()
    )
    daily_total['daily_volume'] = (daily_total['day_end'] - daily_total['day_start']).clip(lower=0)
    return daily_total

@st.cache_data()
def get_weekly_volume(df_tidy):
    # Ensure datetime is datetime type
    df_tidy['datetime'] = pd.to_datetime(df_tidy['datetime'])
    # Group by pump and week
    weekly_total = (
        df_tidy.groupby(['pump', pd.Grouper(key='datetime', freq='W')])['TOTAL_GAL']
        .agg(day_start='first', day_end='last')
        .reset_index()
    )
    # Actual pumped volume = last - first
    weekly_total['weekly_volume'] = (weekly_total['day_end'] - weekly_total['day_start']).clip(lower=0)
    return weekly_total



df1 = run_functions(live_urls[0], col)
df2 = run_functions(live_urls[1], col2)

df_combined = merge_and_sort(df1, df2)
df_tidy = make_tidy(df_combined)

df_tidy['datetime'] = pd.to_datetime(df_tidy['datetime'])
# Set datetime as index
df_tidy_indexed = df_tidy.set_index('datetime')
# Group by pump and resample weekly
weekly = (df_tidy_indexed.groupby('pump')['TOTAL_GAL'].resample('W')  # weekly frequency
    .agg(first='first', last='last')
    .reset_index()  # bring 'pump' back as column
)   


weekly['weekly_volume'] = (weekly['last'] - weekly['first']).clip(lower=0)


2026-04-05 23:47:16.626 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 23:47:16.628 No runtime found, using MemoryCacheStorageManager
2026-04-05 23:47:16.630 No runtime found, using MemoryCacheStorageManager
2026-04-05 23:47:16.631 No runtime found, using MemoryCacheStorageManager
2026-04-05 23:47:16.632 No runtime found, using MemoryCacheStorageManager
2026-04-05 23:47:16.633 No runtime found, using MemoryCacheStorageManager


In [16]:

pump_options1 = sorted(df_tidy['pump'].unique())
selected_pumps1 = st.multiselect("Select Pumps", pump_options1, default=[pump_options1[0]])


min_date = weekly['datetime'].min().date()
max_date = weekly['datetime'].max().date()
start_date, end_date = st.slider(
    "Select date range",
    min_value=min_date,
    max_value=max_date,
    value=(min_date, max_date)
)

# Filter the data
weekly_filtered1 = weekly[
    (weekly['pump'].isin(selected_pumps1)) &
    (weekly['datetime'].dt.date >= start_date) &
    (weekly['datetime'].dt.date <= end_date)
]

# Plot
line_chart = alt.Chart(weekly_filtered1).mark_line(point=True).encode(
    x='datetime:T',
    y='weekly_volume:Q',
    color='pump:N',
    tooltip=['pump', 'datetime', 'weekly_volume']
).interactive()

line_chart

2026-04-05 23:47:16.827 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 23:47:16.828 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 23:47:16.828 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 23:47:16.829 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 23:47:16.829 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 23:47:16.830 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 23:47:16.831 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-05 23:47:16.831 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

alt.Chart(...)

In [17]:
print(df_tidy.columns)

Index(['datetime', 'pump', 'GPM', 'TOTAL_GAL'], dtype='str')


In [21]:
avg_gpm  = df_tidy.groupby('pump', as_index=False)['GPM'].mean().round(1)

# df_tidy.groupby(['well_id', 'datetime'])['gpm'].mean().reset_index()
# df_tidy.set_index('datetime').groupby('well_id')['gpm'].resample('W').mean()

print(avg_gpm)

    pump   GPM
0      1  25.3
1      2  21.3
2      3  16.0
3      4  10.2
4      5  10.7
5      6   6.0
6      7   2.1
7      8   1.1
8      9   6.0
9     10   2.3
10    11   1.7
11    12   3.1
12    13   0.6
13    14   2.9
14    15   1.3
15    16   0.1
16    17   2.2
17    18   0.6
18    19   0.3
19    20   0.7
20    21   1.0
21    22   0.4
22    23   3.7
23    24   4.0
24    25   1.8
25    26   2.8
26    27   3.3
27    28   0.3
28    29   0.1
29    30  12.0
30    31  19.1
31    32  17.8
32    33  11.4
33    34   0.2
34    35   0.0
35    36   0.0
36    37   0.0
37    38  14.0
38    39   4.9
39    40   0.1
40    41   0.0
41    42   0.7
42    43   0.2
43    44   0.5
44    45   0.0
45   101  11.6
46   102  25.0
47   103  21.0
48   104  19.1
49   105   5.6
50   106   1.1
51   107  10.9


In [64]:
avg = df_tidy.groupby(['pump'])['pump'].mean().reset_index().round(1)
print(avg)


ValueError: cannot insert pump, already exists

In [36]:
volumes  = df_tidy.groupby('pump', as_index=False)['GPM'].sort_values['datetime'].iloc[-1]

print(volumes)

AttributeError: 'SeriesGroupBy' object has no attribute 'sort_values'

In [ ]:
z = df_tidy.set_index('datetime').groupby('well_id')['gpm'].resample('W').mean()


In [58]:
latest_gpm = list(df_tidy.loc[df_tidy.groupby('pump')['datetime'].idxmax()]['GPM'])

print(type(latest_gpm))

<class 'list'>
